In [44]:
import json
import re
import math
import srsly
import pandas as pd
from pathlib import Path
from rank_bm25 import BM25Okapi
from beir.retrieval.evaluation import EvaluateRetrieval

In [ ]:
# =========================
# PATHS
# =========================

QUESTIONS_FILE = Path("../QandA_100.jsonl")

PATIENT_QUERIES_FILE = Path("../Task4_Cancer_Patients/Updated_PAR_cancer/new_PAR_queries/new_cancer_train_queries.jsonl")
ARTICLE_CORPUS_FILE = Path("../Task4_Cancer_Patients/Updated_PAR_cancer/new_cancer_corpus.jsonl")

OUTPUT_DIR = Path("hipporag_retrieval_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TOP_K = 5

In [ ]:
# ============================
# Prepare benchmark questions
# ============================

# Load and basic cleaning
questions = pd.DataFrame(list(srsly.read_jsonl(QUESTIONS_FILE)))
questions["patient_id"] = questions["patient_id"].astype(str)
questions["article_id"] = questions["article_id"].astype(str)

# Filter invalid questions
questions = questions[
    questions["question"].notna()
    & (questions["question"].astype(str).str.strip() != "")
    & (~questions["question"].astype(str).str.contains("error", case=False, na=False))
].copy()

# Vectorized ID generation
questions["question_id"] = [f"q_{i:04d}" for i in range(len(questions))]

print(f"Usable questions: {questions.shape}")

Usable questions: (106, 7)


In [52]:
# =========================
# Prepare lookup mappings
# =========================

def process_corpus(path, include_title=True):
    df = pd.DataFrame(list(srsly.read_jsonl(path)))
    df["_id"] = df["_id"].astype(str)
    df["text"] = df["text"].fillna("")
    
    if include_title:
        df["title"] = df.get("title", "").fillna("")
        df["retrieval_text"] = df["title"] + " " + df["text"]
    else:
        df["retrieval_text"] = df["text"]
    
    return df

# Process articles
articles = process_corpus(ARTICLE_CORPUS_FILE, include_title=True)
article_id_to_text = articles.set_index("_id")["retrieval_text"].to_dict()
article_id_to_title = articles.set_index("_id")["title"].to_dict()
article_ids = articles["_id"].tolist()

# Process patients
patients = process_corpus(PATIENT_QUERIES_FILE, include_title=False)
patient_id_to_text = patients.set_index("_id")["retrieval_text"].to_dict()
patient_ids = patients["_id"].tolist()

print(f"Articles: {articles.shape} | Patients: {patients.shape}")

Articles: (398228, 4) | Patients: (54810, 3)


In [53]:
def tokenize(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return text.split()

In [54]:
# =========================
# BM25 Indexing
# =========================

# Tokenize corpus text
articles["tokenized"] = articles["retrieval_text"].apply(tokenize)
patients["tokenized"] = patients["retrieval_text"].apply(tokenize)


# Initialize BM25 indexes using the tokenized document lists
article_bm25 = BM25Okapi(articles["tokenized"].tolist())
patient_bm25 = BM25Okapi(patients["tokenized"].tolist())

In [9]:
def bm25_retrieve(query, bm25_index, doc_ids, top_k=5):
    tokenized_query = tokenize(query)
    scores = bm25_index.get_scores(tokenized_query)

    top_indices = scores.argsort()[::-1][:top_k]

    results = []
    for rank, idx in enumerate(top_indices, start=1):
        results.append({
            "rank": rank,
            "id": doc_ids[idx],
            "score": float(scores[idx])
        })

    return results

In [55]:
# =========================
# BM25 Retrieval 
# =========================

retrieval_results = []

for _, row in questions.iterrows():
    question_id = row["question_id"]
    question = row["question"]

    gold_article_id = row["article_id"]
    gold_patient_id = row["patient_id"]

    retrieved_articles = bm25_retrieve(
        query=question,
        bm25_index=article_bm25,
        doc_ids=article_ids,
        top_k=TOP_K
    )

    retrieved_patients = bm25_retrieve(
        query=question,
        bm25_index=patient_bm25,
        doc_ids=patient_ids,
        top_k=TOP_K
    )

    retrieval_results.append({
        "question_id": question_id,
        "question": question,
        "gold_article_id": gold_article_id,
        "gold_patient_id": gold_patient_id,
        "short_answer": row.get("short_answer", ""),
        "long_answer": row.get("long_answer", ""),
        "retrieved_articles": retrieved_articles,
        "retrieved_patients": retrieved_patients
    })

print("Retrieval complete:", len(retrieval_results))


# Save retrieval results to JSONL
retrieval_output_file = OUTPUT_DIR / "bm25_PAR_retrieval_results_top5.jsonl"
srsly.write_jsonl(retrieval_output_file, retrieval_results)
print("Saved:", retrieval_output_file)

Retrieval complete: 106


In [58]:
# ===========================
# Context Enrichment for RAG
# ===========================
rag_contexts = []

for item in retrieval_results:
    article_contexts = []

    for r in item["retrieved_articles"]:
        aid = r["id"]
        article_contexts.append({
            "rank": r["rank"],
            "article_id": aid,
            "score": r["score"],
            "title": article_id_to_title.get(aid, ""),
            "text": article_id_to_text.get(aid, "")
        })

    patient_contexts = []

    for r in item["retrieved_patients"]:
        pid = r["id"]
        patient_contexts.append({
            "rank": r["rank"],
            "patient_id": pid,
            "score": r["score"],
            "text": patient_id_to_text.get(pid, "")
        })

    rag_contexts.append({
        "question_id": item["question_id"],
        "question": item["question"],
        "gold_short_answer": item["short_answer"],
        "gold_long_answer": item["long_answer"],
        "gold_article_id": item["gold_article_id"],
        "gold_patient_id": item["gold_patient_id"],
        "retrieved_articles": article_contexts,
        "retrieved_patients": patient_contexts
    })

# Save context results to JSONL
rag_context_file = OUTPUT_DIR / "bm25_PAR_rag_contexts_top5.jsonl"
srsly.write_jsonl(rag_context_file, rag_contexts)
print("Saved:", rag_context_file)

Saved: hipporag_retrieval_outputs/bm25_PAR_rag_contexts_top5.jsonl


In [59]:
def evaluate_retrieval(results, gold_key, retrieved_key, label, k):
    qrels = {
        str(res["question_id"]): {str(res[gold_key]): 1} 
        for res in results
    }
    
    run = {
        str(res["question_id"]): {str(doc["id"]): float(doc["score"]) for doc in res[retrieved_key]}
        for res in results
    }

    ndcg, _map, recall, precision = EvaluateRetrieval.evaluate(qrels, run, [k])
    
    # Output
    print(f"--- {label} ---")
    print(f"nDCG@{k}:  {ndcg[f'NDCG@{k}']:.4f}")
    print(f"Recall@{k}: {recall[f'Recall@{k}']:.4f}\n")

In [ ]:
# ===============================
# Evaluate Retrieval Performance
# ===============================

evaluate_retrieval(retrieval_results, "gold_article_id", "retrieved_articles", "ARTICLES", TOP_K)
evaluate_retrieval(retrieval_results, "gold_patient_id", "retrieved_patients", "PATIENTS", TOP_K)

--- ARTICLES ---
nDCG@5:  0.1529
Recall@5: 0.2076

--- PATIENTS ---
nDCG@5:  0.6813
Recall@5: 0.7641

